<center>
  <font color='#023F7C' size="6.5"><b>Hi!ckathon #5 - AI & Sustainability</b></font> 
  <br>
  <font color="#023F7C" size="4"><b>Submission Notebook</b></font>
  <br>
  <br><br>

  <img src="https://www.hi-paris.fr/wp-content/uploads/2020/09/logo-hi-paris-retina.png" width="300" height="200">
</center>

<br>

<ul>
  <li><b>Yanis Montacer</b></li>
  <li><b>Gabriel Gaslain</b></li>
  <li><b>Gabriel Enthoven</b></li>
</ul>

## 1. Libraries and Setup

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb

import optuna
from optuna.samplers import TPESampler

/opt/miniconda3/envs/ml-project-financial-predictions/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data Loading

In [2]:

X = pd.read_csv("X_train.csv")
y = pd.read_csv("y_train.csv")

X = X.drop(columns=["Unnamed: 0"], errors="ignore")
y = y.drop(columns=["Unnamed: 0"], errors="ignore")

## 4. Feature Encoding & Splitting

In [3]:
math_cols = [c for c in X.columns if c.startswith("math_q") and c != "math_q1_total_timing"]
X = X.drop(columns=math_cols, errors="ignore")

In [4]:
cols_to_remove = [
    "Year","CNTRYID","CNTSCHID","CNTSTUID","CYC","NatCen","STRATUM","SUBNATIO","OECD","ADMINMODE","LANGTEST_QQQ","LANGTEST_COG","LANGTEST_PAQ","Option_CT","Option_FL","Option_ICTQ","Option_WBQ","Option_PQ","Option_TQ","Option_UH","BOOKID","COBN_S","PA167","WB153","WB154","WB155","WB160","WB162","WB168","WB177","WB178","ST038","ST036","IC170","IC173","IC176","IC177","IC175","IC183","IC182","IC180"
]

X = X.drop(columns=cols_to_remove, errors="ignore")

In [5]:
obj_cols = X.select_dtypes(include="object").columns
for col in obj_cols:
    X[col] = X[col].astype("category")

cat_cols = X.select_dtypes(include="category").columns

X_lgb = X.copy()
for col in cat_cols:
    X_lgb[col] = X_lgb[col].cat.codes

In [6]:
X_train_full, X_test, X_lgb_train_full, X_lgb_test, y_train_full, y_test = train_test_split(
    X, X_lgb, y, test_size=0.2, random_state=42, shuffle=True
)

y_train_full_vec = y_train_full.values.ravel()
y_test_vec       = y_test.values.ravel()


X_tune, _, X_lgb_tune, _, y_tune, _ = train_test_split(
    X_train_full, X_lgb_train_full, y_train_full,
    test_size=0.7, random_state=123, shuffle=True
)

y_tune_vec = y_tune.values.ravel()

Taille tuning : (281300, 225)  | Taille train_full : (937668, 225)


## 5. XGBoost Optimization

In [7]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 1400),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 5, 10),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 3.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 3.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
        "tree_method": "hist",
        "enable_categorical": True,
        "objective": "reg:squarederror",
        "random_state": 42
    }

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tune, y_tune_vec, test_size=0.2, random_state=42
    )

    model = XGBRegressor(**params)
    model.fit(X_tr, y_tr)

    preds = model.predict(X_val)
    preds = np.clip(preds, 0, None)

    return r2_score(y_val, preds)

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)

study_xgb.optimize(objective_xgb, n_trials=15)

print("XGB - Best score (tuning) :", study_xgb.best_value)
print("XGB - Best params :", study_xgb.best_params)

[I 2025-11-30 04:43:47,439] A new study created in memory with name: no-name-d1a313c0-04ea-4f0e-ba3e-7d28952d4c88
[I 2025-11-30 04:44:20,652] Trial 0 finished with value: 0.7614584729439087 and parameters: {'n_estimators': 837, 'learning_rate': 0.1358198535217987, 'max_depth': 9, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'reg_alpha': 0.46798356100860794, 'reg_lambda': 0.17425083650459838, 'min_child_weight': 7}. Best is trial 0 with value: 0.7614584729439087.
[I 2025-11-30 04:44:38,326] Trial 1 finished with value: 0.7720399997756505 and parameters: {'n_estimators': 1041, 'learning_rate': 0.08329844231669223, 'max_depth': 5, 'subsample': 0.9909729556485983, 'colsample_bytree': 0.9497327922401265, 'reg_alpha': 0.6370173320348285, 'reg_lambda': 0.5454749016213019, 'min_child_weight': 2}. Best is trial 1 with value: 0.7720399997756505.
[I 2025-11-30 04:45:00,116] Trial 2 finished with value: 0.7738253816943241 and parameters: {'n_estimators': 774, 'learning_

XGB - Best score (tuning) : 0.7770871018039346
XGB - Best params : {'n_estimators': 1392, 'learning_rate': 0.0214002921660987, 'max_depth': 8, 'subsample': 0.7030427412812235, 'colsample_bytree': 0.9238913745880993, 'reg_alpha': 1.4556841436630004, 'reg_lambda': 2.9980873159246535, 'min_child_weight': 8}


### 5.1 Train Best XGBoost

In [8]:
best_xgb_params = study_xgb.best_params
best_xgb_params.update({
    "tree_method": "hist",
    "enable_categorical": True,
    "objective": "reg:squarederror",
    "random_state": 42
})

xgb_opt = XGBRegressor(**best_xgb_params)
xgb_opt.fit(X_train_full, y_train_full_vec)

xgb_opt_pred = xgb_opt.predict(X_test)
xgb_opt_pred = np.clip(xgb_opt_pred, 0, None)

print("XGB OPT R² :", r2_score(y_test_vec, xgb_opt_pred))
print("XGB OPT RMSE :", np.sqrt(mean_squared_error(y_test_vec, xgb_opt_pred)))

XGB OPT R² : 0.7844728592598152
XGB OPT RMSE : 56.657356214415294


## 6. LightGBM Optimization

In [10]:
def objective_lgb(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.7, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.7, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "max_depth": -1,
        "verbose": -1
    }

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_lgb_tune, y_tune_vec, test_size=0.2, random_state=42
    )

    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set   = lgb.Dataset(X_val, label=y_val)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=1000,
        valid_sets=[val_set],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(0)
        ]
    )

    preds = model.predict(X_val)
    preds = np.clip(preds, 0, None)

    return r2_score(y_val, preds)


study_lgb = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)

study_lgb.optimize(objective_lgb, n_trials=15)

print("LGB - Best score (tuning) :", study_lgb.best_value)
print("LGB - Best params :", study_lgb.best_params)

[I 2025-11-30 04:57:45,593] A new study created in memory with name: no-name-2a66ff2a-f9d8-4458-a7d0-515a28f54c3c


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[884]	valid_0's rmse: 57.7855


[I 2025-11-30 04:58:38,016] Trial 0 finished with value: 0.7767120404124603 and parameters: {'learning_rate': 0.02757359293934948, 'num_leaves': 244, 'feature_fraction': 0.9195981825434215, 'bagging_fraction': 0.8795975452591109, 'bagging_freq': 2, 'min_data_in_leaf': 48}. Best is trial 0 with value: 0.7767120404124603.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 57.7823


[I 2025-11-30 04:59:41,575] Trial 1 finished with value: 0.7767297846398612 and parameters: {'learning_rate': 0.011703388679635262, 'num_leaves': 225, 'feature_fraction': 0.8803345035229626, 'bagging_fraction': 0.9124217733388136, 'bagging_freq': 1, 'min_data_in_leaf': 195}. Best is trial 1 with value: 0.7767297846398612.


...

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[998]	valid_0's rmse: 57.9074


[I 2025-11-30 05:08:14,630] Trial 13 finished with value: 0.7758455572095516 and parameters: {'learning_rate': 0.015402995594955827, 'num_leaves': 126, 'feature_fraction': 0.8312144244592657, 'bagging_fraction': 0.9372507394663836, 'bagging_freq': 2, 'min_data_in_leaf': 193}. Best is trial 12 with value: 0.7781838892593995.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 57.7876


[I 2025-11-30 05:09:14,374] Trial 14 finished with value: 0.7767035868051609 and parameters: {'learning_rate': 0.014210361509455517, 'num_leaves': 203, 'feature_fraction': 0.8551011954732787, 'bagging_fraction': 0.8107793662044466, 'bagging_freq': 2, 'min_data_in_leaf': 145}. Best is trial 12 with value: 0.7781838892593995.


LGB - Best score (tuning) : 0.7781838892593995
LGB - Best params : {'learning_rate': 0.016837815856581128, 'num_leaves': 254, 'feature_fraction': 0.8418142019973845, 'bagging_fraction': 0.9263939829317358, 'bagging_freq': 2, 'min_data_in_leaf': 200}


In [11]:
best_lgb_params = study_lgb.best_params
best_lgb_params.update({
    "objective": "regression",
    "metric": "rmse",
    "max_depth": -1,
    "verbose": -1
})

train_full_lgb = lgb.Dataset(X_lgb_train_full, label=y_train_full_vec)
val_lgb = lgb.Dataset(X_lgb_test, label=y_test_vec)

lgb_opt = lgb.train(
    best_lgb_params,
    train_full_lgb,
    num_boost_round=2000,
    valid_sets=[val_lgb],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(0)
    ]
)

lgb_opt_pred = lgb_opt.predict(X_lgb_test)
lgb_opt_pred = np.clip(lgb_opt_pred, 0, None)

print("LGB OPT R² :", r2_score(y_test_vec, lgb_opt_pred))
print("LGB OPT RMSE :", np.sqrt(mean_squared_error(y_test_vec, lgb_opt_pred)))

Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 56.4043
LGB OPT R² : 0.7865028629294712
LGB OPT RMSE : 56.389903242748574


## 7. CatBoost Optimization

In [12]:
cat_idx_full = [X_train_full.columns.get_loc(c) for c in cat_cols]

def objective_cat(trial):
    params = {
        "depth": trial.suggest_int("depth", 6, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "border_count": trial.suggest_int("border_count", 64, 255),
        "iterations": trial.suggest_int("iterations", 800, 2000),
        "loss_function": "RMSE",
        "random_state": 42,
        "verbose": False
    }

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tune, y_tune_vec, test_size=0.2, random_state=42
    )

    model = CatBoostRegressor(**params)
    model.fit(X_tr, y_tr, cat_features=cat_idx_full, verbose=False)

    preds = model.predict(X_val)
    preds = np.clip(preds, 0, None)

    return r2_score(y_val, preds)

study_cat = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)

study_cat.optimize(objective_cat, n_trials=15)

print("CAT - Best score (tuning) :", study_cat.best_value)
print("CAT - Best params :", study_cat.best_params)

[I 2025-11-30 05:16:42,106] A new study created in memory with name: no-name-74e2d24a-e506-44f9-ab97-711f78a6c3b3
[I 2025-11-30 05:17:23,315] Trial 0 finished with value: 0.7748060958601042 and parameters: {'depth': 7, 'learning_rate': 0.08927180304353628, 'l2_leaf_reg': 7.587945476302646, 'border_count': 178, 'iterations': 987}. Best is trial 0 with value: 0.7748060958601042.
[I 2025-11-30 05:18:26,596] Trial 1 finished with value: 0.7631433062601002 and parameters: {'depth': 6, 'learning_rate': 0.011430983876313222, 'l2_leaf_reg': 8.795585311974417, 'border_count': 179, 'iterations': 1650}. Best is trial 0 with value: 0.7748060958601042.
[I 2025-11-30 05:19:07,160] Trial 2 finished with value: 0.7739404366669693 and parameters: {'depth': 6, 'learning_rate': 0.09330606024425668, 'l2_leaf_reg': 8.491983767203795, 'border_count': 104, 'iterations': 1018}. Best is trial 0 with value: 0.7748060958601042.
[I 2025-11-30 05:19:52,282] Trial 3 finished with value: 0.7655421351980354 and param

CAT - Best score (tuning) : 0.77574866005564
CAT - Best params : {'depth': 8, 'learning_rate': 0.048925649656201595, 'l2_leaf_reg': 9.973372006531376, 'border_count': 215, 'iterations': 1604}


In [13]:
best_cat_params = study_cat.best_params
best_cat_params.update({
    "loss_function": "RMSE",
    "random_state": 42,
    "verbose": False
})

cat_opt = CatBoostRegressor(**best_cat_params)
cat_opt.fit(X_train_full, y_train_full_vec, cat_features=cat_idx_full, verbose=False)

cat_opt_pred = cat_opt.predict(X_test)
cat_opt_pred = np.clip(cat_opt_pred, 0, None)

print("CAT OPT R² :", r2_score(y_test_vec, cat_opt_pred))
print("CAT OPT RMSE :", np.sqrt(mean_squared_error(y_test_vec, cat_opt_pred)))

CAT OPT R² : 0.7818750886389708
CAT OPT RMSE : 56.99778193846695


## 8. Ensemble Methods: Weighted Blending

In [14]:
best_r2 = -999
best_w = None

for w_xgb in [0.2, 0.3, 0.4, 0.5]:
    for w_cat in [0.2, 0.3, 0.4]:
        w_lgb = 1 - w_xgb - w_cat
        if w_lgb < 0 or w_lgb > 0.6:
            continue

        blend = (
            w_xgb * xgb_opt_pred +
            w_cat * cat_opt_pred +
            w_lgb * lgb_opt_pred
        )
        blend = np.clip(blend, 0, None)
        r2 = r2_score(y_test_vec, blend)

        if r2 > best_r2:
            best_r2 = r2
            best_w = (w_xgb, w_cat, w_lgb)

print("BEST WEIGHTS :", best_w)
print("BEST BLEND R² :", best_r2)

BEST WEIGHTS : (0.3, 0.2, 0.49999999999999994)
BEST BLEND R² : 0.7874255186968809


## 9. Stacking (Meta-Model)

In [15]:
meta_X = np.column_stack([xgb_opt_pred, cat_opt_pred, lgb_opt_pred])
meta_y = y_test_vec

ridge = Ridge(alpha=1.0)
ridge.fit(meta_X, meta_y)

ridge_pred = ridge.predict(meta_X)
ridge_pred = np.clip(ridge_pred, 0, None)

print("Ridge STACK R² :", r2_score(meta_y, ridge_pred))
print("Ridge STACK RMSE :", np.sqrt(mean_squared_error(meta_y, ridge_pred)))

Ridge STACK R² : 0.7876395114720001
Ridge STACK RMSE : 56.23959434302977
